In [ ]:
import torch
from torch.utils.data import Dataset
import os
from PIL import Image
import xmltodict
from torchvision.transforms import ToTensor


# 自定义加载VOC图像的DataLoader
class YOLODataset(Dataset):
    def __init__(
        self, image_folder, label_folder, transform=None, target_transform=None
    ):
        self.image_folder = image_folder
        self.label_folder = label_folder
        self.transform = transform
        self.target_transform = target_transform
        self.img_names = os.listdir(self.image_folder)  # 图像文件名列表
        # print(self.image_folder)  # '../../yolo_data/HelmetDataset-VOC/train/images'
        # print(self.img_names)  # ['new1.png', 'new10.png', 'new100.png', 'new101.png' ...]
        # self.class_list = ["no helmet", "motor", "number", "with helmet"]

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, index):
        img_name = self.img_names[index]
        img_path = os.path.join(self.image_folder, img_name)
        print(img_path)
        image = Image.open(img_path)
        print(image)
        label_name = img_name.split(".")[0] + ".txt"
        label_path = os.path.join(self.label_folder, label_name)
        with open(label_path, "r", encoding="utf-8") as f:
            label_content = f.read()  # 读取yolo label数据，为xml格式
            print(label_content)
            objects = label_content.strip().split("\n") # 去除头尾空格，按照换行符分割
            print(objects) # ['2 0.615894 0.6193279999999999 0.258278 0.07563', '3 0.577815 0.136134 0.274834 0.127731', '0 0.445364 0.602521 0.824503 0.603361']
            target = []
            for object_info in objects:
                info_list = object_info.split(" ")
                class_id = int(info_list[0]) # 类别id
                x_center, y_center, width, height = map(float, info_list[1:]) # 中心坐标和宽高
                target.extend([class_id, x_center, y_center, width, height])
            print(target)
            if self.transform:
                image = self.transform(image) # 图像转换为tensor，3通道
        return image, target


if __name__ == "__main__":
    # 测试数据集
    image_folder = "../../yolo_data/HelmetDataset-YOLO-Train/images"
    label_folder = "../../yolo_data/HelmetDataset-YOLO-Train/labels"
    dataset = YOLODataset(image_folder, label_folder, transform=ToTensor())
    print(len(dataset))
    print(dataset[0])

97
../../yolo_data/HelmetDataset-YOLO-Train/images/new2.jpg
<PIL.PngImagePlugin.PngImageFile image mode=RGB size=302x595 at 0x14830F4C0>
2 0.615894 0.6193279999999999 0.258278 0.07563
3 0.577815 0.136134 0.274834 0.127731
0 0.445364 0.602521 0.824503 0.603361

['2 0.615894 0.6193279999999999 0.258278 0.07563', '3 0.577815 0.136134 0.274834 0.127731', '0 0.445364 0.602521 0.824503 0.603361']
[2, 0.615894, 0.6193279999999999, 0.258278, 0.07563, 3, 0.577815, 0.136134, 0.274834, 0.127731, 0, 0.445364, 0.602521, 0.824503, 0.603361]
(tensor([[[0.4431, 0.4667, 0.4627,  ..., 0.8392, 0.8039, 0.5137],
         [0.0235, 0.0235, 0.0196,  ..., 0.0392, 0.0431, 0.0275],
         [0.9922, 0.9882, 0.9882,  ..., 0.3765, 0.3765, 0.3725],
         ...,
         [0.5294, 0.5294, 0.5294,  ..., 0.5765, 0.5765, 0.5765],
         [0.5294, 0.5294, 0.5294,  ..., 0.5765, 0.5765, 0.5765],
         [0.5294, 0.5294, 0.5294,  ..., 0.5765, 0.5765, 0.5765]],

        [[0.2235, 0.2549, 0.2588,  ..., 0.7216, 0.6745, 0.34